# 02 · Preprocesamiento de texto

**Entrada:** `datos/procesados/resenas_filtradas.csv` (generado por el cuaderno 01)
**Salida:** `datos/procesados/resenas_limpias.csv`

## Objetivo
Transformar el texto libre de las reseñas-queja en texto normalizado apto para TF-IDF, **sin perder el
sentido de las negaciones** (clave en quejas: *"no me quedó bien"*, *"did not fit"*).

Pipeline:
1. Unir `Title` + `Review Text`.
2. Minúsculas y **expansión de contracciones antes de tokenizar** (`didn't` → `did not`).
3. Limpieza de caracteres especiales.
4. Tokenización (NLTK).
5. Eliminación de *stopwords* en inglés **conservando explícitamente las negaciones**.
6. Lematización.

> El texto se mantiene en inglés (idioma original) para no perder matices y para que los términos por
> clúster sean interpretables. `random_state` no aplica aquí (proceso determinista).

In [1]:
from pathlib import Path
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Recursos de NLTK necesarios
for recurso in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)


def encontrar_raiz() -> Path:
    actual = Path.cwd()
    for candidata in [actual, *actual.parents]:
        if (candidata / "datos").is_dir() and (candidata / "modelos").is_dir():
            return candidata
    return actual


RAIZ = encontrar_raiz()
RUTA_FILTRADAS = RAIZ / "datos" / "procesados" / "resenas_filtradas.csv"
RUTA_LIMPIAS = RAIZ / "datos" / "procesados" / "resenas_limpias.csv"
print("Entrada:", RUTA_FILTRADAS, "| existe:", RUTA_FILTRADAS.exists())

Entrada: /home/diegootm/Documentos/triaje-quejas-resenas-ecommerce/datos/procesados/resenas_filtradas.csv | existe: True


## 1. Carga y unión de campos de texto

In [2]:
df = pd.read_csv(RUTA_FILTRADAS)
# Por robustez, descartar cualquier índice sin nombre que se hubiera colado
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
print("Dimensiones:", df.shape)

# Unir título y cuerpo de la reseña (el título suele resumir la queja)
df["texto"] = (df["Title"].fillna("") + " " + df["Review Text"].fillna("")).str.strip()
df[["Title", "Review Text", "texto"]].head(3)

Dimensiones: (4675, 15)


,Title,Review Text,texto
0,Some major design flaws,I had such high hopes for this dress and reall...,Some major design flaws I had such high hopes ...
1,Not for the very petite,"I love tracy reese dresses, but this one is no...",Not for the very petite I love tracy reese dre...
2,"Shimmer, surprisingly goes with lots","I ordered this in carbon for store pick up, an...","Shimmer, surprisingly goes with lots I ordered..."


## 2. Expansión de contracciones (antes de tokenizar)

Expandimos las contracciones **antes** de tokenizar y de quitar la puntuación, porque el apóstrofo se
pierde al limpiar y porque queremos que la negación quede como una palabra completa e independiente.
Nótese que `didn't` se expande a **`did not`** (dos palabras, conservando `not`), no a `did` + `n't`.

In [3]:
# Diccionario de contracciones frecuentes en inglés (prioriza las que contienen negación)
CONTRACCIONES = {
    "didn't": "did not", "doesn't": "does not", "don't": "do not",
    "wasn't": "was not", "weren't": "were not", "isn't": "is not", "aren't": "are not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "won't": "will not", "wouldn't": "would not", "can't": "can not", "cannot": "can not",
    "couldn't": "could not", "shouldn't": "should not", "mustn't": "must not",
    "needn't": "need not", "mightn't": "might not", "shan't": "shall not",
    "it's": "it is", "that's": "that is", "there's": "there is", "here's": "here is",
    "i'm": "i am", "i've": "i have", "i'd": "i would", "i'll": "i will",
    "you're": "you are", "we're": "we are", "they're": "they are",
    "he's": "he is", "she's": "she is", "what's": "what is", "let's": "let us",
}

# Patrón que reemplaza cada contracción como palabra completa
_patron_contr = re.compile(r"\b(" + "|".join(map(re.escape, CONTRACCIONES)) + r")\b")


def expandir_contracciones(texto: str) -> str:
    return _patron_contr.sub(lambda m: CONTRACCIONES[m.group(0)], texto)


# Prueba rápida
print(expandir_contracciones("i didn't like it, it wasn't my size and won't fit"))

i did not like it, it was not my size and will not fit


## 3. Stopwords conservando negaciones, limpieza, tokenización y lematización

De la lista de *stopwords* en inglés de NLTK **restamos las negaciones** para no eliminarlas: `not`,
`no`, `never`, `without`, `nor`, etc. También conservamos las formas que NLTK separa cuando una
contracción **no** se expandió (`didn`, `wasn`, `doesn`, …), como red de seguridad. Perder la negación
convertiría *"not good"* en *"good"* e invertiría el sentido de la queja.

In [4]:
lematizador = WordNetLemmatizer()

# Negaciones que SIEMPRE se conservan (aunque sean cortas o estén en la lista de stopwords)
NEGACIONES = {
    "not", "no", "never", "without", "nor", "none", "neither", "cannot",
    # formas que NLTK separa de contracciones si no se expandieron antes:
    "don", "didn", "doesn", "wasn", "weren", "isn", "aren", "hasn", "haven",
    "hadn", "won", "wouldn", "shouldn", "couldn", "mustn", "needn", "mightn", "shan", "ain",
}

# Stopwords en inglés SIN las negaciones
STOPWORDS_EN = set(stopwords.words("english")) - NEGACIONES

_patron_limpieza = re.compile(r"[^a-z\s]")


def preprocesar(texto: str) -> str:
    texto = str(texto).lower()               # minúsculas
    texto = expandir_contracciones(texto)    # contracciones -> forma expandida (antes de tokenizar)
    texto = _patron_limpieza.sub(" ", texto) # quitar dígitos, puntuación y caracteres especiales
    tokens = word_tokenize(texto)            # tokenización con NLTK
    # Quitar stopwords y tokens muy cortos, PERO conservando siempre las negaciones
    tokens = [t for t in tokens if (t not in STOPWORDS_EN and len(t) > 2) or t in NEGACIONES]
    # Lematización
    tokens = [lematizador.lemmatize(t) for t in tokens]
    return " ".join(tokens)


print("Negaciones conservadas:", sorted(NEGACIONES)[:12], "...")
print("Ejemplo:", preprocesar("This dress did not fit me at all, the fabric was cheap!"))

Negaciones conservadas: ['ain', 'aren', 'cannot', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'mightn'] ...


Ejemplo: dress not fit fabric cheap


In [5]:
df["texto_limpio"] = df["texto"].apply(preprocesar)

# Eliminar filas que quedaron vacías tras la limpieza (p. ej. solo stopwords)
antes = len(df)
df = df[df["texto_limpio"].str.strip().str.len() > 0].reset_index(drop=True)
print(f"Filas vacías tras limpieza eliminadas: {antes - len(df)}")
print("Dimensiones finales:", df.shape)

Filas vacías tras limpieza eliminadas: 0
Dimensiones finales: (4675, 17)


## 4. Ejemplos antes/después (verificación de negaciones)

In [6]:
# Elegir ejemplos que contienen negación para comprobar que NO se pierde el sentido
mascara_neg = df["Review Text"].str.contains(r"\b(?:not|no|never|without|n't)\b", case=False, na=False, regex=True)
ejemplos = df[mascara_neg].head(4)
for _, fila in ejemplos.iterrows():
    print("ANTES :", fila["Review Text"][:200])
    print("DESPUÉS:", fila["texto_limpio"][:200])
    print("-" * 90)

ANTES : I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could 
DESPUÉS: major design flaw high hope dress really wanted work initially ordered petite small usual size found outrageously small small fact could not zip reordered petite medium overall top half comfortable fi
------------------------------------------------------------------------------------------
ANTES : I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dre
DESPUÉS: not petite love tracy reese dress one not petite foot tall usually wear brand dress pretty package lot dress skirt long full overwhelmed small frame not stranger alteration shortening narrowing skirt 
-----------------------------------------------------------------------

## 5. Guardado del texto limpio

In [7]:
# Conservamos metadatos de producto y señales para los cuadernos 03 y 04
columnas_guardar = [
    "Clothing ID", "Age", "Rating", "Recommended IND", "Division Name",
    "Department Name", "Class Name", "vader_compound", "es_queja",
    "Review Text", "texto", "texto_limpio",
]
columnas_guardar = [c for c in columnas_guardar if c in df.columns]
salida = df[columnas_guardar]

RUTA_LIMPIAS.parent.mkdir(parents=True, exist_ok=True)
salida.to_csv(RUTA_LIMPIAS, index=False, encoding="utf-8")
print(f"Guardadas {len(salida)} reseñas limpias en:\n  {RUTA_LIMPIAS}")
salida.head(3)

Guardadas 4675 reseñas limpias en:
  /home/diegootm/Documentos/triaje-quejas-resenas-ecommerce/datos/procesados/resenas_limpias.csv


,Clothing ID,Age,Rating,Recommended IND,Division Name,Department Name,Class Name,vader_compound,es_queja,Review Text,texto,texto_limpio
0,1077,60,3,0,General,Dresses,Dresses,0.9427,True,I had such high hopes for this dress and reall...,Some major design flaws I had such high hopes ...,major design flaw high hope dress really wante...
1,1080,49,2,0,General,Dresses,Dresses,0.9419,True,"I love tracy reese dresses, but this one is no...",Not for the very petite I love tracy reese dre...,not petite love tracy reese dress one not peti...
2,858,39,4,1,General Petite,Tops,Knits,-0.0909,True,"I ordered this in carbon for store pick up, an...","Shimmer, surprisingly goes with lots I ordered...",shimmer surprisingly go lot ordered carbon sto...


## Conclusiones del cuaderno 02

- Se expandieron las contracciones **antes** de tokenizar, de modo que `didn't` → `did not` conserva la
  negación como palabra independiente.
- Las *stopwords* se eliminaron **preservando explícitamente las negaciones**, verificado con ejemplos
  antes/después: *"did not fit"* mantiene `not`.
- El resultado (`texto_limpio`) se guardó en `datos/procesados/resenas_limpias.csv` junto con los
  metadatos de producto, listo para la vectorización TF-IDF del cuaderno 03.